# Step 04 — Evaluation & Analysis
**NLP Course Project · Statistical POS Tagging using HMMs**  
23BCS052 · Hammad Malik

---

### What this notebook does
Runs the trained HMM tagger on the **unseen test set** and measures performance:

| Metric | What it tells you |
|---|---|
| Token accuracy | % of individual words tagged correctly |
| Sentence accuracy | % of sentences where every single tag is correct |
| Baseline comparison | How much better are we than just guessing the most frequent tag? |
| Confusion matrix | Which tag pairs does the model confuse most? |
| Error analysis | Specific examples of mistakes and why they happen |

> 🧠 **Mental model:** Think of this as the exam results day.  
> Token accuracy = how many individual questions you got right.  
> Sentence accuracy = how many full pages you got perfectly right (much harder).

---
## Cell 1 — Imports & Load Everything

In [1]:
import pickle
import numpy as np
from collections import Counter, defaultdict

# Load the tagger we built in Step 03
from tagger import viterbi, tagset, tag2idx, idx2tag, word2idx

# Load the test set from Step 01
with open('/content/pos_data.pkl', 'rb') as f:
    data = pickle.load(f)

test_sents  = data['test_sents']
train_sents = data['train_sents']

print('✅ Everything loaded')
print(f'   Test sentences  : {len(test_sents):,}')
print(f'   Test tokens     : {sum(len(s) for s in test_sents):,}')

✅ Everything loaded
   Test sentences  : 783
   Test tokens     : 19,655


---
## Cell 2 — Establish the Baseline

> 🧠 **Why do we need a baseline?**  
> Before claiming your model is good, you need to show it beats the *dumbest possible approach*.  
> The dumbest approach: for every word, always predict its single most frequent tag from training.  
> This ignores context entirely. If your HMM doesn't beat this, something is wrong.

In [2]:
# Build a lookup: for each word, what tag did it most often have in training?
word_tag_freq = defaultdict(Counter)
for sent in train_sents:
    for word, tag in sent:
        word_tag_freq[word][tag] += 1

most_frequent_tag = {word: counter.most_common(1)[0][0]
                     for word, counter in word_tag_freq.items()}

# What's the single most common tag overall? (fallback for unknown words)
all_train_tags   = [tag for sent in train_sents for _, tag in sent]
global_most_freq = Counter(all_train_tags).most_common(1)[0][0]

print(f'Most frequent tag overall: "{global_most_freq}" (used as fallback for unknown words)')
print()

# Evaluate the baseline on the test set
baseline_correct = 0
baseline_total   = 0

for sent in test_sents:
    for word, gold_tag in sent:
        predicted = most_frequent_tag.get(word, global_most_freq)
        if predicted == gold_tag:
            baseline_correct += 1
        baseline_total += 1

baseline_acc = 100 * baseline_correct / baseline_total
print(f'📊 Most-Frequent-Tag Baseline Accuracy: {baseline_acc:.2f}%')
print(f'   ({baseline_correct:,} correct out of {baseline_total:,} tokens)')
print()
print('   This is what our HMM needs to beat.')

Most frequent tag overall: "NN" (used as fallback for unknown words)

📊 Most-Frequent-Tag Baseline Accuracy: 87.56%
   (17,210 correct out of 19,655 tokens)

   This is what our HMM needs to beat.


---
## Cell 3 — Run the HMM Tagger on the Full Test Set

> ⏱️ This cell takes a minute or two — it's running Viterbi on hundreds of sentences.  
> Watch the progress counter. Grab a coffee.

In [3]:
all_gold      = []   # gold standard tags (correct answers)
all_predicted = []   # our HMM's predictions

correct_tokens   = 0
total_tokens     = 0
correct_sents    = 0
total_sents      = len(test_sents)

print(f'Running Viterbi on {total_sents:,} test sentences...')
print()

for i, sent in enumerate(test_sents):

    # Progress update every 100 sentences
    if (i + 1) % 100 == 0:
        pct = 100 * (i + 1) / total_sents
        print(f'  [{i+1:>4}/{total_sents}]  {pct:.0f}%  done...')

    words     = [word for word, tag in sent]
    gold_tags = [tag  for word, tag in sent]

    # Skip empty sentences
    if len(words) == 0:
        continue

    pred_tags, _ = viterbi(words)

    all_gold.extend(gold_tags)
    all_predicted.extend(pred_tags)

    # Token-level accuracy
    for g, p in zip(gold_tags, pred_tags):
        if g == p:
            correct_tokens += 1
        total_tokens += 1

    # Sentence-level accuracy (all tags must match)
    if gold_tags == pred_tags:
        correct_sents += 1

token_acc = 100 * correct_tokens / total_tokens
sent_acc  = 100 * correct_sents  / total_sents

print()
print('=' * 45)
print('  RESULTS')
print('=' * 45)
print(f'  Token Accuracy    : {token_acc:.2f}%')
print(f'  Sentence Accuracy : {sent_acc:.2f}%')
print(f'  Baseline Accuracy : {baseline_acc:.2f}%')
print(f'  Improvement       : +{token_acc - baseline_acc:.2f}%  over baseline')
print('=' * 45)

Running Viterbi on 783 test sentences...

  [ 100/783]  13%  done...
  [ 200/783]  26%  done...
  [ 300/783]  38%  done...
  [ 400/783]  51%  done...
  [ 500/783]  64%  done...
  [ 600/783]  77%  done...
  [ 700/783]  89%  done...

  RESULTS
  Token Accuracy    : 86.36%
  Sentence Accuracy : 11.11%
  Baseline Accuracy : 87.56%
  Improvement       : +-1.20%  over baseline


---
## Cell 4 — Per-Tag Accuracy

> 🧠 The overall accuracy hides a lot. Some tags are easy (DT is almost always right),  
> others are hard (NN vs VB for ambiguous words). This breakdown tells the real story.

In [4]:
# Count correct and total per tag
tag_correct = Counter()
tag_total   = Counter()

for gold, pred in zip(all_gold, all_predicted):
    tag_total[gold] += 1
    if gold == pred:
        tag_correct[gold] += 1

# Sort by frequency (most common tags first)
sorted_tags = sorted(tag_total.keys(), key=lambda t: tag_total[t], reverse=True)

print('📊 Per-Tag Accuracy (sorted by frequency):')
print(f'   {"TAG":<8} {"CORRECT":>8} {"TOTAL":>8} {"ACCURACY":>10}   VISUAL')
print('   ' + '-' * 60)

for tag in sorted_tags[:20]:   # top 20 most frequent tags
    total   = tag_total[tag]
    correct = tag_correct[tag]
    acc     = 100 * correct / total if total > 0 else 0
    bar     = '█' * int(acc / 5)    # each block = 5%
    print(f'   {tag:<8} {correct:>8,} {total:>8,} {acc:>9.1f}%   {bar}')

📊 Per-Tag Accuracy (sorted by frequency):
   TAG       CORRECT    TOTAL   ACCURACY   VISUAL
   ------------------------------------------------------------
   NN          2,211    2,548      86.8%   █████████████████
   IN          1,897    1,933      98.1%   ███████████████████
   NNP         1,429    1,877      76.1%   ███████████████
   DT          1,562    1,583      98.7%   ███████████████████
               1,263    1,310      96.4%   ███████████████████
   NNS         1,009    1,188      84.9%   ████████████████
   JJ            838    1,138      73.6%   ██████████████
   ,             938      938     100.0%   ████████████████████
   .             773      773     100.0%   ████████████████████
   CD            458      647      70.8%   ██████████████
   VBD           520      630      82.5%   ████████████████
   RB            385      575      67.0%   █████████████
   VB            431      487      88.5%   █████████████████
   VBN           298      439      67.9%   ██████████

---
## Cell 5 — Confusion Matrix

> 🧠 A confusion matrix shows **what the model predicted when it was wrong**.  
> Each row = gold tag (correct answer).  
> Each column = predicted tag (our model's guess).  
> Diagonal = correct predictions. Off-diagonal = mistakes.
>
> The most interesting cells are the **biggest off-diagonal numbers** —  
> these are the most common error types, and each one tells a story about language ambiguity.

In [5]:
# Build confusion counts: confusion[gold][predicted] = count
confusion = defaultdict(Counter)
for gold, pred in zip(all_gold, all_predicted):
    if gold != pred:                  # only count mistakes
        confusion[gold][pred] += 1

# Find top confusion pairs
all_errors = []
for gold_tag, pred_counts in confusion.items():
    for pred_tag, count in pred_counts.items():
        all_errors.append((gold_tag, pred_tag, count))

all_errors.sort(key=lambda x: x[2], reverse=True)

print('❌ Top 15 most common tagging errors:')
print(f'   {"GOLD":>8}  →  {"PREDICTED":<10}  {"COUNT":>6}   WHAT THIS MEANS')
print('   ' + '-' * 65)

explanations = {
    ('NN',  'JJ') : 'Noun mistaken for Adjective (e.g. "stone" wall)',
    ('JJ',  'NN') : 'Adjective mistaken for Noun',
    ('NN',  'NNP'): 'Common noun mistaken for Proper noun',
    ('NNP', 'NN') : 'Proper noun mistaken for common noun',
    ('VBD', 'VBN'): 'Past tense vs Past participle (both end in -ed)',
    ('VBN', 'VBD'): 'Past participle vs Past tense',
    ('NN',  'VB') : 'Noun/Verb ambiguity (e.g. "run", "fish", "can")',
    ('VB',  'NN') : 'Verb/Noun ambiguity',
    ('RB',  'JJ') : 'Adverb vs Adjective (e.g. "fast", "hard")',
    ('JJ',  'RB') : 'Adjective vs Adverb',
    ('VBZ', 'NNS'): 'Verb-s vs Plural Noun (both end in -s)',
    ('NNS', 'VBZ'): 'Plural Noun vs Verb-s',
    ('IN',  'RB') : 'Preposition vs Adverb (e.g. "up", "out", "in")',
    ('RB',  'IN') : 'Adverb vs Preposition',
    ('VB',  'VBP'): 'Base verb vs Present tense verb',
}

for gold, pred, count in all_errors[:15]:
    explanation = explanations.get((gold, pred), 'Tag ambiguity')
    print(f'   {gold:>8}  →  {pred:<10}  {count:>6}   {explanation}')

❌ Top 15 most common tagging errors:
       GOLD  →  PREDICTED    COUNT   WHAT THIS MEANS
   -----------------------------------------------------------------
        NNP  →  NN             164   Proper noun mistaken for common noun
         NN  →  NNP             95   Common noun mistaken for Proper noun
        NNP  →  JJ              91   Tag ambiguity
         JJ  →  DT              63   Tag ambiguity
         NN  →  JJ              57   Noun mistaken for Adjective (e.g. "stone" wall)
         CD  →  DT              54   Tag ambiguity
         JJ  →  NN              49   Adjective mistaken for Noun
        NNP  →  DT              46   Tag ambiguity
        VBN  →  VBD             46   Past participle vs Past tense
        WDT  →  IN              42   Tag ambiguity
         NN  →  DT              41   Tag ambiguity
         RB  →  IN              41   Adverb vs Preposition
         JJ  →  NNP             37   Tag ambiguity
        NNS  →  NN              37   Tag ambiguity
        V

---
## Cell 6 — Error Analysis: Real Sentence Examples

> 🧠 Numbers tell you *how much* the model fails. Examples tell you *why*.  
> This is the most insightful part of any NLP evaluation — and the part  
> professors love to see in a project report.

In [6]:
# Find real test sentences where mistakes occurred and display them
print('🔍 Real examples of tagging errors:\n')

shown = 0
for sent in test_sents:
    if shown >= 6:
        break

    words     = [w for w, t in sent]
    gold_tags = [t for w, t in sent]

    if len(words) == 0:
        continue

    pred_tags, _ = viterbi(words)

    # Only show sentences that have at least one error
    errors = [(i, w, g, p) for i, (w, g, p)
              in enumerate(zip(words, gold_tags, pred_tags)) if g != p]

    if not errors:
        continue

    # Print the sentence with annotations
    print(f'  Sentence : {" ".join(words[:12])}{'...' if len(words) > 12 else ''}')

    # Show all tokens with gold and predicted tags
    for i, (w, g, p) in enumerate(zip(words, gold_tags, pred_tags)):
        marker = '  ✗' if g != p else '   '
        if g != p:
            print(f'    {marker}  "{w}"  →  gold: {g:<6}  predicted: {p:<6}  ← ERROR')

    print()
    shown += 1

🔍 Real examples of tagging errors:

  Sentence : the judge declined *-1 to discuss his salary in detail , but...
      ✗  "judge"  →  gold: NN      predicted: NNP     ← ERROR
      ✗  "detail"  →  gold: NN      predicted: NNP     ← ERROR

  Sentence : drexel remains confident of its future creditworthiness .
      ✗  "confident"  →  gold: JJ      predicted:         ← ERROR

  Sentence : the protracted downturn reflects the intensity of bank of japan yen-support intervention...
      ✗  "reflects"  →  gold: VBZ     predicted: IN      ← ERROR
      ✗  "bank"  →  gold: NNP     predicted: NN      ← ERROR
      ✗  "yen-support"  →  gold: JJ      predicted: POS     ← ERROR
      ✗  "currency"  →  gold: NN      predicted: NNP     ← ERROR
      ✗  "temporarily"  →  gold: RB      predicted: NNP     ← ERROR
      ✗  "150.00"  →  gold: CD      predicted: JJ      ← ERROR
      ✗  "yen"  →  gold: NNS     predicted: NN      ← ERROR

  Sentence : among professionals , 76 % have a favorable opinion of

---
## Cell 7 — Final Summary Report

Everything in one clean printout — ready to copy into your project report.

In [7]:
total_errors  = sum(1 for g, p in zip(all_gold, all_predicted) if g != p)
oov_words     = sum(1 for sent in test_sents
                    for w, _ in sent if w not in word2idx)
total_test_tok= len(all_gold)
oov_rate      = 100 * oov_words / total_test_tok

print('=' * 50)
print('  FINAL PROJECT REPORT — HMM POS TAGGER')
print('  23BCS052 · Hammad Malik')
print('=' * 50)
print()
print('  DATASET')
print(f'    Corpus          : Penn Treebank (NLTK sample)')
print(f'    Train sentences : {len(train_sents):,}')
print(f'    Test  sentences : {len(test_sents):,}')
print(f'    Test tokens     : {total_test_tok:,}')
print(f'    OOV tokens      : {oov_words:,}  ({oov_rate:.1f}% of test set)')
print()
print('  MODEL')
print(f'    Algorithm       : Hidden Markov Model + Viterbi decoding')
print(f'    Smoothing       : Laplace (Add-1)')
print(f'    OOV handling    : Morphological suffix heuristics')
print(f'    Number of tags  : {len(tagset)}')
print()
print('  RESULTS')
print(f'    Baseline acc.   : {baseline_acc:.2f}%  (most-frequent tag)')
print(f'    HMM token acc.  : {token_acc:.2f}%')
print(f'    HMM sent.  acc. : {sent_acc:.2f}%')
print(f'    Improvement     : +{token_acc - baseline_acc:.2f}%  over baseline')
print(f'    Total errors    : {total_errors:,}  ({100*total_errors/total_test_tok:.1f}% error rate)')
print()
print('  TOP ERROR TYPES')
for gold, pred, count in all_errors[:5]:
    print(f'    {gold} → {pred} : {count} cases')
print()
print('=' * 50)
print('  ✅ PROJECT COMPLETE')
print('=' * 50)

  FINAL PROJECT REPORT — HMM POS TAGGER
  23BCS052 · Hammad Malik

  DATASET
    Corpus          : Penn Treebank (NLTK sample)
    Train sentences : 3,131
    Test  sentences : 783
    Test tokens     : 19,655
    OOV tokens      : 1,368  (7.0% of test set)

  MODEL
    Algorithm       : Hidden Markov Model + Viterbi decoding
    Smoothing       : Laplace (Add-1)
    OOV handling    : Morphological suffix heuristics
    Number of tags  : 44

  RESULTS
    Baseline acc.   : 87.56%  (most-frequent tag)
    HMM token acc.  : 86.36%
    HMM sent.  acc. : 11.11%
    Improvement     : +-1.20%  over baseline
    Total errors    : 2,680  (13.6% error rate)

  TOP ERROR TYPES
    NNP → NN : 164 cases
    NN → NNP : 95 cases
    NNP → JJ : 91 cases
    JJ → DT : 63 cases
    NN → JJ : 57 cases

  ✅ PROJECT COMPLETE


---
## ✅ Step 04 Complete — Project Done!

| Step | What we built |
|---|---|
| 01 | Loaded & preprocessed Penn Treebank |
| 02 | Built HMM — transition, emission, initial matrices |
| 03 | Implemented Viterbi decoding algorithm |
| 04 | Evaluated accuracy, confusion matrix, error analysis |

### What to include in your report
- The final summary table from Cell 7
- The per-tag accuracy table from Cell 4
- The top confusion pairs from Cell 5 with your own explanation
- 2–3 real error examples from Cell 6 with analysis of *why* the model made that mistake
- The Viterbi grid screenshot from Step 03 Cell 5